# 00. Kiểm tra môi trường trên Kaggle

Task 0.3, tiêu chí 1: clone repo, cài `requirements.txt`, chạy `scripts/check_env.py`.

**Trước khi Run All:** Settings → Internet: **On**; Accelerator: **GPU T4 x1** (hoặc P100). Không có GPU thì đặt `CPU_ONLY = True`.

**Về torch:** mặc định giữ torch có sẵn của Kaggle (`KEEP_KAGGLE_TORCH = True`) vì cài đè torch 2.14.1 có thể xung đột với torchvision/torchaudio của Kaggle. Khi đó ghi lại phiên bản torch của Kaggle vào `docs/notes_env.md`. Đặt `False` nếu muốn cài đúng `torch==2.14.1+cu126` như local.

Cũng bỏ qua `jupyter`, `ipykernel` (Kaggle tự có, cài đè trong kernel đang chạy dễ làm hỏng kernel).

In [ ]:
REPO_URL = "https://github.com/minucedev/gnn-iot-ids.git"
BRANCH = "main"
KEEP_KAGGLE_TORCH = True
CPU_ONLY = False
REPO = "/kaggle/working/gnn-iot-ids"

In [ ]:
import os, subprocess, sys, time

def run(cmd, **kw):
    """Chạy lệnh, in stdout/stderr ra ô notebook, báo lỗi nếu mã thoát khác 0."""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

if os.path.exists(REPO):
    run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
print(sys.version)
try:
    import torch
    print("torch có sẵn của Kaggle:", torch.__version__, "| CUDA build:", torch.version.cuda)
except ImportError:
    print("Kaggle chưa có torch")

In [ ]:
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
name = lambda r: r.split("==")[0].lower()
torch_pin = next(r for r in reqs if name(r) == "torch")
keep = [r for r in reqs if name(r) not in {"torch", "jupyter", "ipykernel"}]
with open("/tmp/req_kaggle.txt", "w") as f:
    f.write("\n".join(keep) + "\n")
t0 = time.time()
if not KEEP_KAGGLE_TORCH:
    run([sys.executable, "-m", "pip", "install", "-q", torch_pin,
         "--index-url", "https://download.pytorch.org/whl/cu126"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req_kaggle.txt"])
print(f"Cài xong trong {time.time() - t0:.0f} s")

In [ ]:
# Chạy trong tiến trình mới để nạp đúng các gói vừa cài
run([sys.executable, "-I", "scripts/check_env.py"] + (["--cpu-only"] if CPU_ONLY else []), cwd=REPO)

## Sau khi chạy

Chép output của ô trên (kèm phiên bản torch Kaggle, loại GPU, thời gian cài) vào mục **Kaggle** của `docs/notes_env.md`, rồi cập nhật tiêu chí 6 của task 0.3 trong `docs/gnn_ids/TASKS.md`.